# HAKE-MER — Track C: M1+M2+M3 emotion-specific (NRC)

**Modules:** M1 + M2 + M3 with **per-label** NRC priors (GoEmotions↔Plutchik bridge), gate $g$ init 0.

**Compare to:** M1+M2 macro (~0.505 DistilBERT) and legacy **global** M3 NRC.

**Protocol:** batch 16, LR 5e-5, 4 epochs, seeds 42/123/456.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Runtime → Change runtime type → GPU")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import subprocess
from getpass import getpass
from pathlib import Path

REPO, WORKDIR = "khalef-khalil/marii", Path("/content/marii")
PUBLIC_URL = f"https://github.com/{REPO}.git"
TRAIN_FLAGS = "--epochs 4 --batch-size 16 --lr 5e-5 --early-stopping-patience 0"

def clone_repo() -> None:
    if WORKDIR.is_dir():
        subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(WORKDIR), "reset", "--hard", "origin/main"], check=True)
        return
    r = subprocess.run(["git", "clone", "--depth", "1", PUBLIC_URL, str(WORKDIR)], capture_output=True)
    if r.returncode == 0:
        return
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = getpass("GitHub token: ")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git", str(WORKDIR)], check=True)

clone_repo()
%cd {WORKDIR}
!git rev-parse --short HEAD

In [ ]:
!pip install -q -r requirements-train.txt

In [ ]:
!./run_m1_m2_m3_es_campaign.sh nrc --backbone distilbert-base-uncased {TRAIN_FLAGS}

In [ ]:
import json
from pathlib import Path

path = Path("reference/artifacts/m1_m2_m3_nrc_es_distilbert_base_uncased_campaign.json")
c = json.loads(path.read_text(encoding="utf-8"))
print(path.name, "modules:", c.get("modules", {}))
if "lexicon_gate_abs" in c:
    print("  |g| mean:", c["lexicon_gate_abs"]["mean"])
for k, b in c["test_aggregate"].items():
    print(f"  {k}: {b['mean']:.4f} ± {b['std']:.4f}")

After training: run per-class export on each `runs/*_m1_m2_m3_nrc_es` directory and compute $\Delta$ frequent vs rare (same groups as ch. 4). SenticNet ES: duplicate notebook with `run_m1_m2_m3_es_campaign.sh senticnet`.